### get_weather

A script to query the openweather API to collect and transform the data for data visualization.

In [ ]:
import requests
import json
import pandas as pd
import time
import datetime
import pytz
import pygsheets

from pathlib import Path
from functools import reduce

In [ ]:
## API credentials
APPID = '868c135ff7f42fa7e6688c7dba057746'

In [ ]:
# Cleveland Park Coordinates
lat = '38.9346'
lon = '-77.0664'

In [ ]:
units = 'imperial'

In [ ]:
url = f'https://api.openweathermap.org/data/2.5/onecall?lat={lat}&lon={lon}&appid={APPID}&units={units}'

In [ ]:
# Returns a response object
response = requests.get(url)

In [ ]:
# Check for errors. If no exceptions are raised, the downloaded text is stored in response.text
response.raise_for_status()

In [ ]:
# Load JSON data into a Python object
weatherdata = json.loads(response.text)

In [ ]:
weather_data = pd.DataFrame()

In [ ]:
## Use orient index to avoid ValueError: arrays must all be same length
weather_data = weather_data.from_dict(weatherdata, orient='index')

In [ ]:
weather_data = weather_data.transpose()

In [ ]:
## Create a DataFrame for the daily data
daily_df = pd.DataFrame(weather_data.daily[0])

In [ ]:
## Turn columns of dicts into DataFrames
temp_df = pd.DataFrame(list(daily_df.temp))
temp_df.columns = [str(col) + '_temp' for col in temp_df.columns]

In [ ]:
feels_df = pd.DataFrame(list(daily_df.feels_like))
feels_df.columns = [str(col) + '_feel' for col in feels_df.columns]

In [ ]:
weather_df = pd.DataFrame(list(daily_df.weather))
weather_list = list(weather_df[0])
weather_df = pd.DataFrame(weather_list)

In [ ]:
merge_list = [daily_df, temp_df, feels_df, weather_df]

In [ ]:
## merge the list of DataFrames
merge_df = reduce(lambda x, y: pd.merge(x, y, left_index=True, right_index=True), merge_list)

In [ ]:
## Drop columns
merge_df = merge_df.drop(['temp', 'feels_like', 'weather'], axis=1)

In [ ]:
## Convert datetime columns from unix to standard time
merge_df[['dt', 'sunrise', 'sunset']] = merge_df[['dt', 'sunrise', 'sunset']].apply(pd.to_datetime, unit='s')
merge_df = merge_df.rename(columns={'dt': 'date_time'})

In [ ]:
## Make datetime objects offset aware and set time to 'Eastern/US'
utc_tz = pytz.utc
eastern_tz = pytz.timezone('US/Eastern')

merge_df['date_time'] = merge_df['date_time'].dt.tz_localize(utc_tz)
merge_df['sunrise'] = merge_df['sunrise'].dt.tz_localize(utc_tz)
merge_df['sunset'] = merge_df['sunset'].dt.tz_localize(utc_tz)

merge_df['date_time'] = merge_df['date_time'].dt.tz_convert(eastern_tz)
merge_df['sunrise'] = merge_df['sunrise'].dt.tz_convert(eastern_tz)
merge_df['sunset'] = merge_df['sunset'].dt.tz_convert(eastern_tz)

In [ ]:
## Transform colums containing _temp or _feel with pd.melt 
temps = merge_df[['date_time','day_temp', 'min_temp', 'max_temp', 'night_temp', 'eve_temp', 'morn_temp']]
feels = merge_df[['date_time', 'day_feel', 'night_feel', 'eve_feel', 'morn_feel']]

temps = pd.melt(temps, id_vars='date_time', value_vars=['day_temp', 'min_temp', 'max_temp', 'night_temp', 'eve_temp', 'morn_temp'])
feels = pd.melt(feels, id_vars='date_time', value_vars=['day_feel', 'night_feel', 'eve_feel', 'morn_feel'])

temps = temps.rename(columns={'variable': 'temp', 'value': 'temp_value'})
feels = feels.rename(columns={'variable': 'feel', 'value': 'feel_value'})

merge_df = pd.merge(merge_df, temps, left_on='date_time', right_on='date_time')
merge_df = pd.merge(merge_df, feels, left_on='date_time', right_on='date_time')

merge_df = merge_df.drop(['day_feel', 'night_feel', 'eve_feel', 'morn_feel', 'day_temp', 'min_temp', 'max_temp', 'night_temp', 'eve_temp', 'morn_temp'], axis=1)

In [ ]:
## Write data to Google Sheets
print('Writing to Sheets...')
service = pygsheets.authorize('credentials.json')
workbook = service.open('seven_day_forecast')
sheet = workbook[0]
sheet.set_dataframe(merge_df, (1,1), fit=False)